In [3]:
import torch, os, glob, random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy.fftpack import dct as scipy_dct
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

BASE = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'
TRAIN_REAL = os.path.join(BASE, 'train/real')
TRAIN_FAKE = os.path.join(BASE, 'train/fake')
VAL_REAL   = os.path.join(BASE, 'valid/real')
VAL_FAKE   = os.path.join(BASE, 'valid/fake')
TEST_REAL  = os.path.join(BASE, 'test/real')
TEST_FAKE  = os.path.join(BASE, 'test/fake')
MODELS_DIR = '/kaggle/working/'

print("\n📁 Dataset check:")
for name, path in [
    ('Train real', TRAIN_REAL), ('Train fake', TRAIN_FAKE),
    ('Val real',   VAL_REAL),   ('Val fake',   VAL_FAKE),
    ('Test real',  TEST_REAL),  ('Test fake',  TEST_FAKE),
]:
    count = len(os.listdir(path))
    print(f"  ✅ {name:12s} → {count:,} images")

✅ Device: cuda
✅ GPU: Tesla T4

📁 Dataset check:
  ✅ Train real   → 50,000 images
  ✅ Train fake   → 50,000 images
  ✅ Val real     → 10,000 images
  ✅ Val fake     → 10,000 images
  ✅ Test real    → 10,000 images
  ✅ Test fake    → 10,000 images


In [4]:
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussianBlur(p=0.2),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None, max_per_class=10000):
        self.transform = transform
        self.samples   = []
        real_imgs = glob.glob(os.path.join(real_dir, '*.jpg'))[:max_per_class]
        fake_imgs = glob.glob(os.path.join(fake_dir, '*.jpg'))[:max_per_class]
        for p in real_imgs: self.samples.append((p, 1))
        for p in fake_imgs: self.samples.append((p, 0))
        random.shuffle(self.samples)
        print(f"✅ {len(real_imgs)} real + {len(fake_imgs)} fake = {len(self.samples)} total")

    def compute_frequency(self, img_gray):
        fft      = np.fft.fft2(img_gray)
        fft_mag  = np.log(np.abs(np.fft.fftshift(fft)) + 1)
        fft_norm = (fft_mag - fft_mag.min()) / (fft_mag.max() - fft_mag.min() + 1e-8)
        dct_img  = scipy_dct(scipy_dct(img_gray.T, norm='ortho').T, norm='ortho')
        dct_log  = np.log(np.abs(dct_img) + 1)
        dct_norm = (dct_log - dct_log.min()) / (dct_log.max() - dct_log.min() + 1e-8)
        return torch.from_numpy(np.stack([fft_norm, dct_norm], axis=0).astype(np.float32))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img      = np.array(Image.open(img_path).convert('RGB').resize((224,224)))
        img_gray = np.array(Image.open(img_path).convert('L').resize((224,224)), dtype=np.float32) / 255.0
        img_tensor  = self.transform(image=img)['image'] if self.transform else torch.from_numpy(img.transpose(2,0,1)).float()/255.0
        freq_tensor = self.compute_frequency(img_gray)
        return img_tensor, freq_tensor, torch.tensor(label, dtype=torch.long)

print("✅ Dataset class ready!")

✅ Dataset class ready!


In [5]:
class CNNBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0, global_pool='avg')
    def forward(self, x): return self.backbone(x)

class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
    def forward(self, x): return self.net(x).view(x.size(0), -1)

class ViTBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
    def forward(self, x): return self.backbone(x)

class DeepShieldAI(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn_branch  = CNNBranch()
        self.vit_branch  = ViTBranch()
        self.freq_branch = FrequencyBranch()
        self.fusion = nn.Sequential(
            nn.Linear(2560, 1024), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(1024, 512),  nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 128),   nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
    def forward(self, img, freq):
        combined = torch.cat([self.cnn_branch(img), self.vit_branch(img), self.freq_branch(freq)], dim=1)
        return self.fusion(combined)

model = DeepShieldAI().to(device)
print(f"✅ DeepShield AI: {sum(p.numel() for p in model.parameters()):,} parameters")

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

✅ DeepShield AI: 100,097,162 parameters


In [6]:
train_dataset = DeepfakeDataset(TRAIN_REAL, TRAIN_FAKE, transform=train_transform, max_per_class=10000)
val_dataset   = DeepfakeDataset(VAL_REAL,   VAL_FAKE,   transform=val_transform,   max_per_class=3000)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Train batches : {len(train_loader)}")
print(f"✅ Val batches   : {len(val_loader)}")

✅ 10000 real + 10000 fake = 20000 total
✅ 3000 real + 3000 fake = 6000 total
✅ Train batches : 625
✅ Val batches   : 188


In [7]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, freqs, labels in tqdm(loader, desc="Training"):
        imgs, freqs, labels = imgs.to(device), freqs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs, freqs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, freqs, labels in tqdm(loader, desc="Validating"):
            imgs, freqs, labels = imgs.to(device), freqs.to(device), labels.to(device)
            outputs    = model(imgs, freqs)
            loss       = criterion(outputs, labels)
            total_loss += loss.item()
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

EPOCHS   = 5
best_acc = 0.0
history  = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print("🚀 Starting DeepShield AI Training!")
print(f"   Branches : CNN + ViT + Frequency")
print(f"   Epochs   : {EPOCHS}")
print(f"   GPU      : {torch.cuda.get_device_name(0)}")
print("-" * 50)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = val_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        save_path = os.path.join(MODELS_DIR, 'deepshield_final_best.pth')
        torch.save(model.state_dict(), save_path)
        size = os.path.getsize(save_path) / (1024*1024)
        print(f"  💾 Model saved! Size: {size:.1f} MB")

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

print("-" * 50)
print(f"🏆 Training Complete! Best Val Accuracy: {best_acc*100:.2f}%")
print(f"\n📁 Saved files: {os.listdir(MODELS_DIR)}")

🚀 Starting DeepShield AI Training!
   Branches : CNN + ViT + Frequency
   Epochs   : 5
   GPU      : Tesla T4
--------------------------------------------------


Training:  11%|█         | 70/625 [01:41<13:26,  1.45s/it]


KeyboardInterrupt: 

In [ ]:
!pip install -q gradio
print("✅ Gradio installed!")

In [10]:
import gradio as gr
import torch
import torch.nn as nn
import timm
import numpy as np
from PIL import Image
from scipy.fftpack import dct as scipy_dct

# ── Load model ─────────────────────────────────────────────
def load_model():
    model = DeepShieldAI().to(device)
    model.load_state_dict(torch.load('/kaggle/working/deepshield_final_best.pth',
                                      map_location=device))
    model.eval()
    return model

deepshield_model = load_model()
print("✅ Model loaded!")

# ── Preprocessing ──────────────────────────────────────────
def preprocess_image(img):
    # RGB tensor
    img_resized = img.resize((224, 224))
    img_np      = np.array(img_resized).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_norm    = (img_np - mean) / std
    img_tensor  = torch.from_numpy(img_norm.transpose(2,0,1)).float().unsqueeze(0)

    # Frequency tensor
    img_gray = np.array(img.convert('L').resize((224,224)), dtype=np.float32) / 255.0
    fft      = np.fft.fft2(img_gray)
    fft_mag  = np.log(np.abs(np.fft.fftshift(fft)) + 1)
    fft_norm = (fft_mag - fft_mag.min()) / (fft_mag.max() - fft_mag.min() + 1e-8)
    dct_img  = scipy_dct(scipy_dct(img_gray.T, norm='ortho').T, norm='ortho')
    dct_log  = np.log(np.abs(dct_img) + 1)
    dct_norm = (dct_log - dct_log.min()) / (dct_log.max() - dct_log.min() + 1e-8)
    freq_tensor = torch.from_numpy(
        np.stack([fft_norm, dct_norm], axis=0).astype(np.float32)
    ).unsqueeze(0)

    return img_tensor, freq_tensor

# ── Prediction function ────────────────────────────────────
def predict(image):
    if image is None:
        return "Please upload an image!", "", "", ""

    img = Image.fromarray(image).convert('RGB')
    img_tensor, freq_tensor = preprocess_image(img)

    img_tensor  = img_tensor.to(device)
    freq_tensor = freq_tensor.to(device)

    with torch.no_grad():
        output = deepshield_model(img_tensor, freq_tensor)
        probs  = torch.softmax(output, dim=1)[0]

    fake_score = probs[0].item()
    real_score = probs[1].item()

    if fake_score >= 0.55:
        verdict = f"🚨 FAKE DETECTED"
        color   = "red"
    else:
        verdict = f"✅ REAL FACE"
        color   = "green"

    result = f"""
## {verdict}

| Branch | Score |
|--------|-------|
| Fake Probability | {fake_score*100:.1f}% |
| Real Probability | {real_score*100:.1f}% |

**Confidence: {max(fake_score, real_score)*100:.1f}%**
    """
    return result

# ── Gradio UI ──────────────────────────────────────────────
with gr.Blocks(title="DeepShield AI") as demo:
    gr.Markdown("""
    # 🛡️ DeepShield AI
    ### Multi-Domain Deepfake Detection System
    *CNN + Vision Transformer + Frequency Analysis*
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="Upload Face Image")
            detect_btn  = gr.Button("🔍 Detect", variant="primary")

        with gr.Column():
            result_output = gr.Markdown(label="Result")

    detect_btn.click(
        fn      = predict,
        inputs  = [image_input],
        outputs = [result_output]
    )

    gr.Markdown("""
    ### How it works:
    - **CNN Branch** — Detects texture and blending artifacts
    - **ViT Branch** — Checks global facial consistency
    - **Frequency Branch** — Finds hidden GAN fingerprints
    """)

demo.launch(share=True, debug=False)

✅ Model loaded!
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://73faf5ff3124411cb6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2202, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1979, in postprocess_data
    prediction_value = block.postprocess(prediction_value)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/components/markdown.py", line 146, in postproc

In [9]:
import os
save_path = '/kaggle/working/deepshield_final_best.pth'
torch.save(model.state_dict(), save_path)
size = os.path.getsize(save_path) / (1024*1024)
print(f"✅ Model saved! Size: {size:.1f} MB")
print(f"📁 Files: {os.listdir('/kaggle/working/')}")

✅ Model saved! Size: 382.5 MB
📁 Files: ['deepshield_final_best.pth', '.virtual_documents']
